## Running Karstolution for the Philippines
Taken from:
https://github.com/swasc/karstolution/tree/master

In [ ]:
from __future__ import print_function, division, unicode_literals
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
%matplotlib inline

In [ ]:
# this lets us use karstolution without installing it
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
from Karstolution import karstolution

In [ ]:
# Static configurations that the ca be tuned. 
# Make changes to parameters here for the Philippines, BNBNP cave system:

yaml_config = """
f1 : 0.2
f3 : 0.008
f4 : 0.0
f5 : 0.005
f6 : 0.002
f7 : 1.0
k_diffuse : 0.008
f8 : 0.001
i : 0.5
j : 0.25
k : 0.25
m : 0.75
n : 0.25
k_eevap : 0.0
k_d18o_soil : 0.03
k_d18o_epi : 0.0
soilstore : 200.0
epicap : 400.0
ovicap : 100.0
epikarst : 400.0
ks1 : 400.0
ks2 : 200.0
lambda_weibull : 1.5
k_weibull : 1.0
weibull_delay_months: 6
mixing_parameter_phi : 1.0
monthly_forcing : 
  # these parameters are forced with a climatological mean
  drip_pco2 : [11738.59677,4322.196428,1423.83871,1707.546296,1574.252688,1568.166666,20610.42593,38161.07079,40123.91389,30843.12455,23197.40833,17095.08059]
  cave_pco2 : [5869.298387,2161.098214,711.919355,853.773148,787.126344,784.083333,10305.21296,19080.53539,20061.95694,15421.56228,11598.70417,8547.540294]
  rel_humidity : [0.89,0.89,0.85,0.85,0.85,0.85,0.85,0.85,0.89,0.89,0.89,0.89]
  ventilation : [0.2,0.2,0.1,0.1,0.1,0.1,0.1,0.1,0.2,0.2,0.2,0.2]
  cave_temp : [25.58,25.49,25.55,25.73,25.88,25.62,25.59,25.71,25.71,25.76,25.73,25.71]
  drip_interval : [100.0,100.0,100.0,100.0,5s0.0,50.0,50.0,50.0,50.0,50.0,100.0,100.0]
  driprate_store_empty : [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0]
  driprate_store_full :  [0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01]
initial_conditions :
  # initial level in each store
  soil : 50.0
  epikarst : 100.0
  ks1 : 230.0
  ks2 : 50.0
  diffuse : 30.0
  # initial oxygen-18 composition in each store (d18O, permille VSMOW)
  d18o_soil : -7.51
  d18o_epikarst : -7.51
  d18o_ks1 : -7.05
  d18o_ks2 : -7.05
  d18o_prevrain : -3.71
  d18o_diffuse : -7.05"""

In [ ]:
config=yaml.safe_load(yaml_config)

## Time-varying input

The time-varying inputs (with a monthly timestep) are passed to the model as a pandas.DataFrame.  Load one from a csv file...

In [ ]:
# Updating this to the phil monthly forcing
df_input = pd.read_csv('../notebooks/Phil-monthly-forcing2.csv')
df_input.head()

In [ ]:
# Plotting the monthly forcing
ax = df_input.drop(['tt','mm'], axis=1).plot(subplots=True)
ax[0].figure.set_size_inches([5,10])
ax[0].set_title('Model forcing')

In [ ]:
# checking if everything looks good
display(df_input.head())
display(df_input.tail())
display(df_input.shape)
display(df_input.dtypes)
#check to see any NANs present?
display(df_input.isna().sum())

display(df_input[['evpt', 'prp', 'tempp', 'd18o']].describe())

In [ ]:
# Running Model 
model_output = karstolution(config, df_input, calculate_drip=True)
print(model_output.columns)

## Plot results

In [ ]:
# Drip Interval Plot; higher y-value = longer time bw drips
drip_cols = [itm for itm in model_output.columns if itm.startswith('drip')]
model_output[drip_cols].plot()

In [ ]:
# Modeled calcite d180 values 
stal_cols = [itm for itm in model_output.columns if itm.startswith('stal') and '18o' in itm]
model_output[stal_cols].plot()

In [ ]:
# Modeled Growth Rate mm/year
stal_cols = [itm for itm in model_output.columns if itm.startswith('stal') and 'growth_rate' in itm]
model_output[stal_cols].plot()

In [ ]:
# Water-depth(storage) in mm 
store_cols = ['soilstor', 'epxstor', 'kststor1','kststor2']
model_output[store_cols].plot()

In [ ]:
# Fluxes all in mm/month
flux_cols = ['f1', 'f3', 'f4', 'f5', 'f6', 'f7']
model_output[flux_cols].plot()

In [ ]:
# Fluxes small in mm/month
small_flux_cols = ['f3', 'f4', 'f5', 'f6', 'f7']
model_output[small_flux_cols].plot()

In [ ]:
model_output['dpdf[0]'].plot()
ax = model_output['dpdf[0]'].plot()
ax.set_xlabel('Model timestep (months)')
ax.set_title('New input to delayed epikarst-flow pathway')
ax.set_ylabel('New delayed-flow input (mm/month)')

In [ ]:
# Isotope calcite plots
fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(df_input['d18o'].to_numpy(),
        label='Precipitation δ18O')

ax.plot(model_output['soil18o'].to_numpy(),
        label='Soil water δ18O')

ax.plot(model_output['epx18o'].to_numpy(),
        label='Epikarst water δ18O')

ax.plot(model_output['kststor118o'].to_numpy(),
        label='KS1 water δ18O')

ax.plot(model_output['kststor218o'].to_numpy(),
        label='KS2 water δ18O')

ax.set_xlabel('Model timestep (months)')
ax.set_ylabel('Water δ18O (‰)')
ax.set_title('Filtering of precipitation δ18O through the karst system')
ax.legend()

plt.show()

In [ ]:
# isolates the hydrological isotope signal immediately before isotope_calcite().
rain = df_input['d18o'].to_numpy()
rain_previous = df_input['d18o'].shift(1).to_numpy()

ks1 = model_output['kststor118o'].to_numpy()

# Same equations used inside karst_process
drip118o = (
    config['i'] * ks1
    + config['j'] * rain
    + config['k'] * rain_previous
)

drip218o = (
    config['m'] * ks1
    + config['n'] * rain
)

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(model_output['kststor218o'],
        label='Stal1 source water (KS2)')

ax.plot(drip118o,
        label='Stal2 source water (bypass mix)')

ax.plot(drip218o,
        label='Stal3 source water (bypass mix)')

ax.plot(model_output['epx18o'],
        label='Stal4 source water (epikarst)')

ax.plot(model_output['kststor118o'],
        label='Stal5 source water (KS1)')

ax.set_xlabel('Model timestep (months)')
ax.set_ylabel('Source-water δ18O (‰)')
ax.set_title('δ18O of water supplied to each modeled stalagmite')
ax.legend()

plt.show()

In [ ]:
# a direct source-water δ¹⁸O vs. calcite δ¹⁸O comparison for all five stalagmites.
# ----------------------------
# Reconstruct source-water δ18O
# ----------------------------

rain = df_input['d18o'].to_numpy()
rain_previous = df_input['d18o'].shift(1).to_numpy()

ks1 = model_output['kststor118o'].to_numpy()

# Stal2 and Stal3 bypass mixtures
drip118o = (
    config['i'] * ks1
    + config['j'] * rain
    + config['k'] * rain_previous
)

drip218o = (
    config['m'] * ks1
    + config['n'] * rain
)

# Source water supplied to isotope_calcite()
source_water = {
    'Stal1': model_output['kststor218o'].to_numpy(),
    'Stal2': drip118o,
    'Stal3': drip218o,
    'Stal4': model_output['epx18o'].to_numpy(),
    'Stal5': model_output['kststor118o'].to_numpy()
}

# Corresponding modeled calcite δ18O
calcite = {
    'Stal1': model_output['stal1d18o'].to_numpy(),
    'Stal2': model_output['stal2d18o'].to_numpy(),
    'Stal3': model_output['stal3d18o'].to_numpy(),
    'Stal4': model_output['stal4d18o'].to_numpy(),
    'Stal5': model_output['stal5d18o'].to_numpy()
}

# ----------------------------
# Plot
# ----------------------------

fig, axes = plt.subplots(5, 1, figsize=(11, 14), sharex=True)

for ax, name in zip(axes, source_water.keys()):

    ax.plot(source_water[name],
            label='Source-water δ18O')

    ax.plot(calcite[name],
            label='Calcite δ18O')

    ax.set_ylabel('δ18O (‰)')
    ax.set_title(name)
    ax.legend()

axes[-1].set_xlabel('Model timestep (months)')

fig.suptitle(
    'Source-water δ18O versus modeled calcite δ18O',
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# aside: COMPARING MODEL DRIP RATE and OBSERVED DRIP RATE INTERVALS

obs_monthly = pd.read_csv(
    "observed_monthly_drip.csv",
    index_col=0,
    parse_dates=True
)

obs_monthly_interval = obs_monthly["interval_sec"]

model_output.index = pd.date_range(
    start="2021-01-01",
    end="2022-12-01",
    freq="MS"
)

overlap = obs_monthly_interval.index.intersection(
    model_output.index
)

print(overlap)

comparison = pd.DataFrame(index=overlap)

comparison["Observed"] = obs_monthly_interval.loc[overlap]

comparison["Stal1"] = model_output.loc[overlap, "drip_int_stal1"]
comparison["Stal2"] = model_output.loc[overlap, "drip_int_stal2"]
comparison["Stal3"] = model_output.loc[overlap, "drip_int_stal3"]
comparison["Stal4"] = model_output.loc[overlap, "drip_int_stal4"]
comparison["Stal5"] = model_output.loc[overlap, "drip_int_stal5"]

plt.figure(figsize=(10, 5))

plt.plot(
    comparison.index,
    comparison["Observed"],
    marker="o",
    linewidth=3,
    label="Observed fast drip"
)

for col in ["Stal1", "Stal2", "Stal3", "Stal4", "Stal5"]:
    plt.plot(
        comparison.index,
        comparison[col],
        marker="o",
        label=col
    )

plt.ylabel("Drip interval (seconds/drop)")
plt.xlabel("Date")
plt.title("Observed vs modeled drip interval")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

### Plots for HESS paper #

## 1. First make one master diagnostic figure

In [ ]:
# Read observed drip data
drip_fast = pd.read_csv("stal_fast.csv")
drip_slow = pd.read_csv("stal_slow.csv")

# Convert Time to datetime and make it the index
drip_fast["Time"] = pd.to_datetime(drip_fast["Time"])
drip_slow["Time"] = pd.to_datetime(drip_slow["Time"])

drip_fast = drip_fast.set_index("Time")
drip_slow = drip_slow.set_index("Time")

# Convert raw drip rates to seconds/drop
drip_fast["drip_interval"] = (
    60 / drip_fast["fast_drops/minute"]
)

# Zero slow-drip rate means no dripping.
# Do NOT divide zero by 60; set these intervals to NaN for plotting.
drip_slow["drip_interval"] = np.where(
    drip_slow["slow_drops/minute"] > 0,
    60 / drip_slow["slow_drops/minute"],
    np.nan
)

# Monthly mean rate converted to interval
drip_fast_monthly_rate = (
    drip_fast["fast_drops/minute"]
    .resample("MS")
    .mean()
)
drip_fast_monthly_interval = 60 / drip_fast_monthly_rate

drip_slow_monthly_rate = (
    drip_slow["slow_drops/minute"]
    .resample("MS")
    .mean()
)

drip_slow_monthly_interval = 60 / drip_slow_monthly_rate


In [ ]:
# Make sure forcing and model output use the same dates
forcing = df_input.copy()
forcing.index = model_output.index

fig, axes = plt.subplots(
    4, 1,
    figsize=(12, 12),
    sharex=True
)

# --------------------------------
# 1. Precipitation
# --------------------------------
axes[0].bar(
    forcing.index,
    forcing["prp"],
    width=20
)
axes[0].set_ylabel("Precipitation\n(mm/month)")
axes[0].set_title("Philippines Karstolution diagnostic — 2021–2022")


# --------------------------------
# 2. Water stores
# --------------------------------
axes[1].plot(
    model_output.index,
    model_output["soilstor"],
    label="Soil"
)
axes[1].plot(
    model_output.index,
    model_output["epxstor"],
    label="Epikarst"
)
axes[1].plot(
    model_output.index,
    model_output["kststor1"],
    label="KS1"
)
axes[1].plot(
    model_output.index,
    model_output["kststor2"],
    label="KS2"
)

axes[1].set_ylabel("Storage (mm)")
axes[1].legend(ncol=4)


# --------------------------------
# 3. Reservoir water d18O
# --------------------------------
axes[2].plot(
    model_output.index,
    model_output["soil18o"],
    label="Soil"
)
axes[2].plot(
    model_output.index,
    model_output["epx18o"],
    label="Epikarst"
)
axes[2].plot(
    model_output.index,
    model_output["kststor118o"],
    label="KS1"
)
axes[2].plot(
    model_output.index,
    model_output["kststor218o"],
    label="KS2"
)

axes[2].set_ylabel("Water δ18O (‰)")
axes[2].legend(ncol=4)


# --------------------------------
# 4. MODELED Drip intervals
# --------------------------------

axes[3].plot(
    model_output.index,
    model_output["drip_int_stal1"],
    label="Stal1"
)
axes[3].plot(
    model_output.index,
    model_output["drip_int_stal2"],
    label="Stal2"
)
axes[3].plot(
    model_output.index,
    model_output["drip_int_stal3"],
    label="Stal3"
)
axes[3].plot(
    model_output.index,
    model_output["drip_int_stal4"],
    label="Stal4"
)
axes[3].plot(
    model_output.index,
    model_output["drip_int_stal5"],
    label="Stal5"
)

# ---------------------------------------------------------
# RAW OBSERVED DRIP INTERVALS
# ---------------------------------------------------------

axes[3].scatter(
    drip_fast.index,
    drip_fast["drip_interval"],
    s=8,
    alpha=0.3,
    label="Observed fast drip",
    zorder=10
)

axes[3].scatter(
    drip_slow.index,
    drip_slow["drip_interval"],
    s=8,
    alpha=0.3,
    label="Observed slow drip",
    zorder=10
)

# ---------------------------------------------------------
# MONTHLY OBSERVED DRIP INTERVALS
# ---------------------------------------------------------

axes[3].scatter(
    drip_slow_monthly_rate.index,
    drip_slow_monthly_interval,
    s=80, 
    alpha=0.3, 
    marker= 'D',
    label= "Observed slow drip (monthly)",
    zorder=10
)

axes[3].scatter(
    drip_fast_monthly_rate.index,
    drip_fast_monthly_interval,
    s=80, 
    alpha=0.3, 
    marker= 'D',
    label= "Observed fast drip (monthly)",
    zorder=10
)

axes[3].set_ylim(1, 500)
axes[3].set_ylabel("Drip interval\n(seconds/drop)")
axes[3].set_xlabel("Date")
axes[3].legend(ncol=5)

plt.tight_layout()

plt.savefig(
    "Philippines_Karstolution_diagnostic.eps",
    format="eps",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

### This figure answers a fundamental question:

When precipitation changes, which parts of the modeled karst respond rapidly and which remain buffered?

# 2. Then examine precipitation δ¹⁸O propagation

Figure showing precipitation δ¹⁸O and the four reservoir-water isotope compositions:

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    forcing.index,
    forcing["d18o"],
    marker="o",
    label="Precipitation"
)

ax.plot(
    model_output.index,
    model_output["soil18o"],
    label="Soil"
)

ax.plot(
    model_output.index,
    model_output["epx18o"],
    label="Epikarst"
)

ax.plot(
    model_output.index,
    model_output["kststor118o"],
    label="KS1"
)

ax.plot(
    model_output.index,
    model_output["kststor218o"],
    label="KS2"
)

ax.set_ylabel("Water δ18O (‰ VSMOW)")
ax.set_xlabel("Date")
ax.set_title("Propagation of precipitation δ18O through the karst")

ax.legend()
plt.tight_layout()

plt.savefig(
    "Philippines_Karstolution_d18o_precip.eps",
    format="eps",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# 3. Quantify that buffering


In [ ]:
isotope_summary = pd.DataFrame({
    "Precipitation": forcing["d18o"],
    "Soil": model_output["soil18o"],
    "Epikarst": model_output["epx18o"],
    "KS1": model_output["kststor118o"],
    "KS2": model_output["kststor218o"]
})

summary = pd.DataFrame({
    "Mean": isotope_summary.mean(),
    "SD": isotope_summary.std(),
    "Min": isotope_summary.min(),
    "Max": isotope_summary.max(),
    "Range": isotope_summary.max() - isotope_summary.min()
})

summary.round(3)

### 4. Then connect that result to the real fast drip

# First: plot modeled calcite δ¹⁸O

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    model_output.index,
    model_output["stal1d18o"],
    marker="o",
    label="Stal1"
)

ax.plot(
    model_output.index,
    model_output["stal2d18o"],
    marker="o",
    label="Stal2"
)

ax.plot(
    model_output.index,
    model_output["stal3d18o"],
    marker="o",
    label="Stal3"
)

ax.plot(
    model_output.index,
    model_output["stal4d18o"],
    marker="o",
    label="Stal4"
)

ax.plot(
    model_output.index,
    model_output["stal5d18o"],
    marker="o",
    label="Stal5"
)

ax.set_ylabel("Calcite δ18O (‰ VPDB)")
ax.set_xlabel("Date")
ax.set_title("Modeled calcite δ18O — Stal2 and Stal3")

ax.legend()
plt.tight_layout()

plt.savefig(
    "Philippines_Karstolution_modeled_calcite.eps",
    format="eps",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# Two - calculate a growth-weighted modeled calcite δ¹⁸O over each actual plate interval.

In [ ]:
def plate_weighted_d18o(model, start, end, d18o_col, growth_col):
    """
    Calculate modeled calcite d18O over an actual plate deployment
    interval, weighted by modeled calcite growth and days represented
    by each monthly timestep.
    """

    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    weighted_sum = 0.0
    total_weight = 0.0

    for date, row in model.iterrows():

        # interval represented by this monthly model value
        month_start = date
        month_end = date + pd.offsets.MonthBegin(1)

        # overlap between model month and plate deployment
        overlap_start = max(month_start, start)
        overlap_end = min(month_end, end)

        days = (overlap_end - overlap_start).days

        if days > 0:
            growth = row[growth_col]
            d18o = row[d18o_col]

            weight = growth * days

            weighted_sum += d18o * weight
            total_weight += weight

    return weighted_sum / total_weight

In [ ]:
# plotting modeled growth rate plate and observed plate 

plates = [
    ("Plate 1", "2022-08-13", "2022-10-13", -9.761),
    ("Plate 2", "2022-10-13", "2022-12-01", -9.326),
]

results = []

for name, start, end, observed in plates:

    stal2 = plate_weighted_d18o(
        model_output,
        start,
        end,
        "stal2d18o",
        "stal2_growth_rate"
    )

    stal3 = plate_weighted_d18o(
        model_output,
        start,
        end,
        "stal3d18o",
        "stal3_growth_rate"
    )

    results.append({
        "Plate": name,
        "Observed": observed,
        "Stal2": stal2,
        "Stal3": stal3
    })

plate_comparison = pd.DataFrame(results).set_index("Plate")

plate_comparison.round(3)

# Le Fin #

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ---------------------------------------------------------
# OBSERVED FAST-DRIP and SLOW-DRIP WATER d18O from water-stable
# ---------------------------------------------------------

# FAST DRIP
obs_fast_d18o = pd.Series(
    {
        "2022-08-12": -7.8403,
        "2022-09-09": -7.6996,
        "2022-10-14": -7.9673,
        "2022-11-17": -7.642,
        "2022-12-13": -7.896,
    }
)

obs_fast_d18o.index = pd.to_datetime(obs_fast_d18o.index)

# SLOW DRIP  
obs_slow_d18o = pd.Series(
    {
        "2022-08-12": -6.945,
        "2022-09-09": -7.667,
        "2022-10-14": -7.912,
        "2022-11-17": -7.8477,
        "2022-12-13": -7.863,
    }
)

obs_slow_d18o.index = pd.to_datetime(obs_slow_d18o.index)

# ---------------------------------------------------------
# MODELED STAL2 / STAL3 SOURCE WATER
#
# These are the values from your diagnostic output.
# ---------------------------------------------------------

stal2_water = pd.Series(
    {
        "2022-08-01": -7.7405,
        "2022-09-01": -7.5582,
        "2022-10-01": -8.4725,
        "2022-11-01": -9.3483,
        "2022-12-01": -7.3001,
    }
)

stal3_water = pd.Series(
    {
        "2022-08-01": -7.3132,
        "2022-09-01": -7.3461,
        "2022-10-01": -8.2287,
        "2022-11-01": -8.2262,
        "2022-12-01": -6.1877,
    }
)

stal2_water.index = pd.to_datetime(stal2_water.index)
stal3_water.index = pd.to_datetime(stal3_water.index)


# ---------------------------------------------------------
# OBSERVED CALCITE PLATES taken from ISO_FRAC_CALC
# ---------------------------------------------------------

plate1_start = pd.Timestamp("2022-08-13")
plate1_end   = pd.Timestamp("2022-10-13")
slow1_d18o  = -9.761
fast1_d18o  = -9.664

plate2_start = pd.Timestamp("2022-10-13")
plate2_end   = pd.Timestamp("2022-12-01")
slow2_d18o  = -9.466
fast2_d18o  = -9.326


# ---------------------------------------------------------
# FIGURE
# ---------------------------------------------------------

fig, axes = plt.subplots(
    3, 1,
    figsize=(12, 11),
    sharex=True
)


# =========================================================
# PANEL A — PRECIPITATION d18O
# =========================================================

ax = axes[0]

ax.plot(
    forcing.index,
    forcing["d18o"],
    marker="o",
    linewidth=1.5,
    label="Precipitation"
)

ax.set_ylabel("Precipitation δ18O\n(‰ VSMOW)")
ax.set_title("A. Precipitation isotope forcing")
ax.legend()


# =========================================================
# PANEL B — DRIP-WATER d18O
# =========================================================

ax = axes[1]

ax.plot(
    model_output.index,
    model_output["kststor218o"],
    linewidth=2,
    label="Stal1 (KS2)"
)

ax.plot(
    model_output.index,
    model_output["stal2_source_water_d18o"],
    marker="o",
    linewidth=2,
    label="Modeled Stal2 source water"
)

ax.plot(
    model_output.index,
    model_output["stal3_source_water_d18o"],
    marker="o",
    linewidth=2,
    label="Modeled Stal3 source water"
)

ax.plot(
    model_output.index,
    model_output["epx18o"],
    linewidth=2,
    label="Stal4 (epikarst)"
)

ax.plot(
    model_output.index,
    model_output["kststor118o"],
    linewidth=2,
    label="Stal5 (KS1)"
)

# Observations as points rather than a connected line
ax.scatter(
    obs_fast_d18o.index,
    obs_fast_d18o,
    s=80,
    marker="o",
    label="Observed fast-drip water",
    zorder=5
)

ax.scatter(
    obs_slow_d18o.index,
    obs_fast_d18o,
    s=40,
    marker='s',
    label='Observed slow-drip water',
    zorder=5
)

ax.set_ylabel("Drip-water δ18O\n(‰ VSMOW)")
ax.set_title("B. Modeled and observed fast-drip water")
ax.legend()


# =========================================================
# PANEL C — CALCITE d18O
# =========================================================

ax = axes[2]

ax.plot(
    model_output.index,
    model_output["stal1d18o"],
    marker="o",
    label="Stal1"
)

ax.plot(
    model_output.index,
    model_output["stal2d18o"],
    marker="o",
    label="Stal2"
)

ax.plot(
    model_output.index,
    model_output["stal3d18o"],
    marker="o",
    label="Stal3"
)

ax.plot(
    model_output.index,
    model_output["stal4d18o"],
    marker="o",
    label="Stal4"
)

ax.plot(
    model_output.index,
    model_output["stal5d18o"],
    marker="o",
    label="Stal5"
)

# Plate 1 Fast: horizontal line spanning deployment period
ax.hlines(
    slow1_d18o,
    plate1_start,
    plate1_end,
    linewidth=4,
    label="Observed Slow Plate 1"
)

# Plate 2 Fast 
ax.hlines(
    fast1_d18o,
    plate1_start,
    plate1_end,
    linewidth=4,
    alpha=0.5,
    label="Observed Fast Plate 2"
)

ax.hlines(
    slow2_d18o,
    plate2_start,
    plate2_end,
    linewidth=4,
    label="Observed Slow Plate 1"
)

# Plate 2 Fast 
ax.hlines(
    fast2_d18o,
    plate2_start,
    plate2_end,
    linewidth=4,
    alpha=0.5,
    label="Observed Fast Plate 2"
)

# Mark plate boundaries
for date in [plate1_start, plate1_end, plate2_end]:
    ax.axvline(
        date,
        linestyle="--",
        linewidth=0.8,
        alpha=0.5
    )

ax.set_ylabel("Calcite δ18O\n(‰ VPDB)")
ax.set_xlabel("Date")
ax.set_title("C. Modeled calcite and observed plate calcite")
ax.legend(ncol=2)


# ---------------------------------------------------------
# Include 2022 for comparison...
# ---------------------------------------------------------

axes[2].set_xlim(
    pd.Timestamp("2022-01-01"),
    pd.Timestamp("2022-12-31")
)

axes[2].xaxis.set_major_locator(
    mdates.MonthLocator(interval=1)
)

axes[2].xaxis.set_major_formatter(
    mdates.DateFormatter("%b")
)

plt.tight_layout()

plt.savefig(
    "Philippines_Karstolution_synthesis.eps",
    format="eps",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
## Internal diagnostic tests for trouble shooting karstolution
store_comparison = model_output.loc[
    overlap,
    [
        "soilstor",
        "epxstor",
        "kststor1",
        "kststor2",
        "drip_int_stal1",
        "drip_int_stal2",
        "drip_int_stal3",
        "drip_int_stal4",
        "drip_int_stal5"
    ]
].copy()

store_comparison["Observed_interval"] = (
    obs_monthly_interval.loc[overlap]
)

store_comparison["Observed_rate"] = (
    1 / store_comparison["Observed_interval"]
)

print(store_comparison)

########### NEW SECTION ###############

forcing_overlap = df_input.copy()

# Give forcing data its real dates
forcing_overlap.index = pd.date_range(
    start="2021-01-01",
    end="2022-12-01",
    freq="MS"
)

diagnostic = pd.DataFrame(index=overlap)

diagnostic["Precip_mm"] = forcing_overlap.loc[overlap, "prp"]
diagnostic["ET_mm"] = forcing_overlap.loc[overlap, "evpt"]

diagnostic["Observed_interval"] = (
    obs_monthly_interval.loc[overlap]
)

diagnostic["Observed_rate"] = (
    1 / diagnostic["Observed_interval"]
)

diagnostic["Soil"] = model_output.loc[overlap, "soilstor"]
diagnostic["Epikarst"] = model_output.loc[overlap, "epxstor"]
diagnostic["KS1"] = model_output.loc[overlap, "kststor1"]
diagnostic["KS2"] = model_output.loc[overlap, "kststor2"]

diagnostic["f1"] = model_output.loc[overlap, "f1"]
diagnostic["f3"] = model_output.loc[overlap, "f3"]
diagnostic["f4"] = model_output.loc[overlap, "f4"]
diagnostic["f5"] = model_output.loc[overlap, "f5"]
diagnostic["f6"] = model_output.loc[overlap, "f6"]
diagnostic["f7"] = model_output.loc[overlap, "f7"]

print(diagnostic)

########### NEW SECTION ###############

obs_monthly = pd.read_csv(
    "observed_monthly_drip.csv",
    index_col=0,
    parse_dates=True
)

print(obs_monthly)

########### NEW SECTION ###############

cols = [
    "soilstor", "epxstor", "kststor1", "kststor2",
    "f1", "f3", "f4", "f5", "f6", "f7"
]

print(
    model_output[cols].describe().T[
        ["min", "mean", "max"]
    ]
)

print(
    "\nMonths at epikarst capacity:",
    (model_output["epxstor"] >= 399.99).sum(),
    "out of", len(model_output)
)

print(
    "Months at KS1 capacity:",
    (model_output["kststor1"] >= 399.99).sum(),
    "out of", len(model_output)
)

########### NEW SECTION ###############

# Give the forcing data its correct monthly dates
forcing = df_input.copy()

forcing.index = pd.date_range(
    start="2021-01-01",
    periods=len(forcing),
    freq="MS"
)

# Antecedent precipitation
forcing["P_current"] = forcing["prp"]
forcing["P_lag1"] = forcing["prp"].shift(1)
forcing["P_lag2"] = forcing["prp"].shift(2)

# Simple cumulative antecedent precipitation
forcing["P_current_plus_lag1"] = (
    forcing["P_current"] + forcing["P_lag1"]
)

forcing["P_previous_2mo"] = (
    forcing["P_lag1"] + forcing["P_lag2"]
)

forcing["P_3mo"] = (
    forcing["P_current"]
    + forcing["P_lag1"]
    + forcing["P_lag2"]
)

########### NEW SECTION ###############

lag_test = forcing.loc[
    overlap,
    [
        "P_current",
        "P_lag1",
        "P_lag2",
        "P_current_plus_lag1",
        "P_previous_2mo",
        "P_3mo"
    ]
].copy()

lag_test["Observed_interval"] = (
    obs_monthly_interval.loc[overlap]
)

lag_test["Observed_rate"] = (
    1 / lag_test["Observed_interval"]
)

lag_test["Observed_drops_min"] = (
    60 / lag_test["Observed_interval"]
)

print(lag_test)

########### NEW SECTION ###############

# Add several lagged precipitation series
lag_analysis = pd.DataFrame(index=overlap)

lag_analysis["Observed_rate"] = (
    obs_monthly.loc[overlap, "drops_per_min"]
)

lag_analysis["P_t"] = forcing.loc[overlap, "prp"]

lag_analysis["P_lag1"] = (
    forcing["prp"].shift(1).loc[overlap]
)

lag_analysis["P_lag2"] = (
    forcing["prp"].shift(2).loc[overlap]
)

lag_analysis["P_lag3"] = (
    forcing["prp"].shift(3).loc[overlap]
)

print(lag_analysis)

print("\nCorrelation with observed drip rate:")
print(
    lag_analysis.corr()["Observed_rate"]
    .drop("Observed_rate")
    .sort_values(ascending=False)
)


########### NEW SECTION ###############

hydro_test = model_output.loc[
    overlap,
    [
        "soilstor",
        "epxstor",
        "kststor1",
        "kststor2",
        "f1",
        "f3",
        "f5",
        "f6",
        "dpdf[0]"
    ]
].copy()

hydro_test["Observed_rate"] = (
    obs_monthly.loc[overlap, "drops_per_min"]
)

print(hydro_test)

print("\nCorrelation with observed drip rate:")

print(
    hydro_test.corr()["Observed_rate"]
    .drop("Observed_rate")
    .sort_values(ascending=False)
)

########### NEW SECTION ###############

delayed = pd.Series(
    [2.9586, 3.0694, 3.1586, 3.2330, 3.2850],
    index=overlap,
    name="Delayed_diffuse"
)

compare_delay = pd.DataFrame({
    "Observed_rate": obs_monthly.loc[overlap, "drops_per_min"],
    "Delayed_diffuse": delayed,
    "P_lag2": forcing["prp"].shift(2).loc[overlap]
})

print(compare_delay)

print("\nCorrelations:")
print(compare_delay.corr()["Observed_rate"])

########### NEW SECTION ###############

from Karstolution.karst_process import weibull_parameters_y

y6 = weibull_parameters_y(
    config["lambda_weibull"],
    config["k_weibull"],
    config["weibull_delay_months"]
)

print(y6)
print("number of weights:", len(y6))
print("sum:", y6.sum())

########### NEW SECTION ###############

delayed6 = pd.Series(
    [3.2249, 3.3513, 3.4370, 3.4799, 3.4714],
    index=overlap,
    name="Delayed_6mo"
)

comparison_weibull = pd.DataFrame({
    "Observed_rate": obs_monthly.loc[overlap, "drops_per_min"],
    "Delayed_12mo": [2.9586, 3.0694, 3.1586, 3.2330, 3.2850],
    "Delayed_6mo": delayed6,
    "P_lag2": forcing["prp"].shift(2).loc[overlap]
})

print(comparison_weibull)
print("\nCorrelation with observed rate:")
print(
    comparison_weibull.corr()["Observed_rate"]
    .sort_values(ascending=False)
)

########### NEW SECTION ###############

comparison = pd.DataFrame({
    "Observed": obs_monthly.loc[overlap, "interval_sec"],
    "Stal1": model_output.loc[overlap, "drip_int_stal1"],
    "Stal2": model_output.loc[overlap, "drip_int_stal2"],
    "Stal3": model_output.loc[overlap, "drip_int_stal3"],
    "Stal4": model_output.loc[overlap, "drip_int_stal4"],
    "Stal5": model_output.loc[overlap, "drip_int_stal5"],
})

print(comparison)

########### NEW SECTION ###############

model_output.loc[
    "2022-08-01":"2022-12-01",
    [
        "stal2d18o",
        "stal3d18o",
        "stal2_growth_rate",
        "stal3_growth_rate"
    ]
]

########### NEW SECTION ###############

model_output.loc[
    "2022-08-01":"2022-12-01",
    ["cave_temp"]
]

########### NEW SECTION ###############

# Observed FAST drip-water d18O
# September was sampled but has no measured d18O value
obs_fast_d18o = pd.Series(
    {
        "2022-08-01": -7.576198,
        "2022-09-01": np.nan,
        "2022-10-01": -7.6410803,
        "2022-11-01": -7.8002207,
        "2022-12-01": -7.576198,
    }
)

obs_fast_d18o.index = pd.to_datetime(obs_fast_d18o.index)

# Modeled source-water d18O from your 6-month run
stal2_water = pd.Series(
    {
        "2022-08-01": -7.7405,
        "2022-09-01": -7.5582,
        "2022-10-01": -8.4725,
        "2022-11-01": -9.3483,
        "2022-12-01": -7.3001,
    }
)

stal3_water = pd.Series(
    {
        "2022-08-01": -7.3132,
        "2022-09-01": -7.3461,
        "2022-10-01": -8.2287,
        "2022-11-01": -8.2262,
        "2022-12-01": -6.1877,
    }
)

stal2_water.index = pd.to_datetime(stal2_water.index)
stal3_water.index = pd.to_datetime(stal3_water.index)

# Combine into one table
water_comparison = pd.DataFrame({
    "Observed_fast": obs_fast_d18o,
    "Stal2": stal2_water,
    "Stal3": stal3_water
})

water_comparison

########### NEW SECTION ###############

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    water_comparison.index,
    water_comparison["Observed_fast"],
    marker="o",
    linewidth=2,
    label="Observed fast drip"
)

ax.plot(
    water_comparison.index,
    water_comparison["Stal2"],
    marker="o",
    label="Modeled Stal2 source water"
)

ax.plot(
    water_comparison.index,
    water_comparison["Stal3"],
    marker="o",
    label="Modeled Stal3 source water"
)

ax.set_ylabel("Water δ18O (‰ VSMOW)")
ax.set_xlabel("Date")
ax.set_title("Observed vs modeled fast-drip water δ18O")
ax.legend()

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()